| Aspect                  | Machine Learning (ML)                                | Deep Learning (DL)                                     |                                                                            |
| ----------------------- | ---------------------------------------------------- | ------------------------------------------------------ | -------------------------------------------------------------------------- |
| **Feature Engineering** | Requires manual feature selection and extraction     | Automatically learns features from raw data            |                                                                            |
| **Data Requirements**   | Performs well with smaller datasets                  | Requires large amounts of data for optimal performance |                                                                            |
| **Model Complexity**    | Utilizes simpler models (e.g., decision trees, SVMs) | Employs complex architectures like neural networks     |                                                                            |
| **Hardware Dependency** | Less computationally intensive                       | Requires high-performance hardware (e.g., GPUs)        |                                                                            |


# Definitions: Batch, Iteration, Epoch

| Term      | Definition                                                  | Example                                                              |
|-----------|-------------------------------------------------------------|------------------------------------------------------------------------------|
| **Batch**     | A subset of the dataset processed at once (e.g., 64 samples) / A subset of the dataset used to train the model in one forward/backward pass. | `X, y = next(iter(dataloader))` gives one batch (`X.shape = [64, 1, 28, 28]`). |
| **Iteration** | One step of training (processing one batch + updating weights). | Each loop in `for batch, (X, y) in enumerate(dataloader)` is one iteration.   |
| **Epoch**     | One full pass through the entire dataset (all batches).     | The outer loop `for t in range(epochs)` runs 5 epochs.                        |

---

###  Example

If the dataset has **60,000 images** and the **batch size is 100**:

- **Batches per epoch** = 60,000 / 100 = **600 iterations**
- **5 epochs** = 5 × 600 = **3,000 total iterations**

---

###  Key Points

- **Batch Size**: Controls memory usage and gradient stability (e.g., 64, 128).
- **Iteration**: One forward/backward pass on a single batch.
- **Epoch**: Ensures the model sees all training data multiple times.


## And gate
| Input 1 | Input 2 | Output |
| ------- | ------- | ------ |
| 0       | 0       | 0      |
| 0       | 1       | 0      |
| 1       | 0       | 0      |
| 1       | 1       | 1      |


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# AND gate data
X = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
y = torch.tensor([[0], [0], [0], [1]], dtype=torch.float32)

# Define simple model: 1 neuron (no hidden layers)
class ANDModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))  # use sigmoid for binary classification

# Initialize model
model = ANDModel()

# Loss and optimizer
loss_fn = nn.BCELoss()  # Binary Cross Entropy for binary output can use another loss as BCEwith logits loss
optimizer = optim.SGD(model.parameters(), lr=0.1) # used sgd as it is simple and effective for this task
# Optimizer can be Adam or any other optimizer as well

# Training loop
epochs = 200
for epoch in range(epochs):
    y_pred = model(X) # Calls model.forward(X)
    loss = loss_fn(y_pred, y)
    optimizer.zero_grad() # Clears old gradients
    loss.backward() # Computes gradients
    optimizer.step() # Updates model weights

# Evaluate
with torch.no_grad():
    predictions = model(X)
    predicted_classes = (predictions > 0.5).float() 
    accuracy = (predicted_classes == y).float().mean()
    print("\nAND Gate Results:")
    print("Predictions:\n", predicted_classes.numpy())
    print("Accuracy:", accuracy.item() * 100)

# Save model
torch.save(model.state_dict(), "and_model.pth")

# Load model
loaded_model = ANDModel()
loaded_model.load_state_dict(torch.load("and_model.pth"))
loaded_model.eval()

# Predict again
with torch.no_grad():
    print("Loaded Model Predictions:")
    print((loaded_model(X) > 0.5).float())



AND Gate Results:
Predictions:
 [[0.]
 [0.]
 [0.]
 [1.]]
Accuracy: 100.0
Loaded Model Predictions:
tensor([[0.],
        [0.],
        [0.],
        [1.]])


## Xor gate
- XOR is not linearly separable, so a single-layer, single-neuron model will not work

| Input 1 | Input 2 | Output |
| ------- | ------- | ------ |
| 0       | 0       | 0      |
| 0       | 1       | 1      |
| 1       | 0       | 1      |
| 1       | 1       | 0      |


In [14]:
# XOR Data
X = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
y = torch.tensor([[0], [1], [1], [0]], dtype=torch.float32)

# Define model with hidden layer
class XORModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4)  # 1 hidden layer with 4 neurons
        self.output = nn.Linear(4, 1)

    def forward(self, x):
        x = torch.relu(self.hidden(x))
        return torch.sigmoid(self.output(x))

# Initialize
xor_model = XORModel()
loss_fn = nn.BCELoss()
optimizer = optim.Adam(xor_model.parameters(), lr=0.01) #Adam optimizer improves convergence speed

# Training
for epoch in range(1000):
    y_pred = xor_model(X)
    loss = loss_fn(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Evaluation
with torch.no_grad():
    predictions = xor_model(X)
    predicted_classes = (predictions > 0.5).float()
    accuracy = (predicted_classes == y).float().mean()
    print("\nXOR Gate Results:")
    print("Predictions:\n", predicted_classes.numpy())
    print("Accuracy:", accuracy.item() * 100)

# Save and Reload
torch.save(xor_model.state_dict(), "xor_model.pth")
loaded_xor = XORModel()
loaded_xor.load_state_dict(torch.load("xor_model.pth"))
loaded_xor.eval()
print("Loaded XOR Model Predictions:")
print((loaded_xor(X) > 0.5).float())



XOR Gate Results:
Predictions:
 [[0.]
 [1.]
 [1.]
 [0.]]
Accuracy: 100.0
Loaded XOR Model Predictions:
tensor([[0.],
        [1.],
        [1.],
        [0.]])
